<a href="https://colab.research.google.com/github/Janith303/Sinhala-and-Maths-Model-training-/blob/Restnet50-new/notebook/resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import json
import hashlib
import time
import random
import platform
import shutil
import subprocess
import gc

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import sklearn
import kagglehub
import matplotlib
import matplotlib.pyplot as plt

# Architecture-specific preprocessor for ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess_input

print("All imports completed successfully.")

All imports completed successfully.


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "TensorFlow op determinism enabled"
except Exception as e:
    determinism_status = f"Could not enable op determinism: {e}"

print(f"Seed:        {SEED}")
print(f"Determinism: {determinism_status}")

Seed:        42
Determinism: TensorFlow op determinism enabled


In [3]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        gpu_name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True,
        ).strip().splitlines()[0]
        nvidia_smi = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
            text=True,
        ).strip().splitlines()[0]
    except Exception:
        gpu_name = tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'Unknown GPU')
        nvidia_smi = 'Unavailable'
else:
    gpu_name = 'No GPU detected'
    nvidia_smi = 'N/A'

environment_info = {
    'member': 3,
    'model': 'ResNet50',
    'python_version': sys.version,
    'platform': platform.platform(),
    'tensorflow_version': tf.__version__,
    'keras_version': keras.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'scikit_learn_version': sklearn.__version__,
    'kagglehub_version': getattr(kagglehub, '__version__', 'unknown'),
    'matplotlib_version': matplotlib.__version__,
    'gpu_name': gpu_name,
    'nvidia_smi': nvidia_smi,
    'seed': SEED,
    'determinism_status': determinism_status,
}

print('Environment Recorded:')
for k, v in environment_info.items():
    print(f'  {k:<22}: {v}')

Environment Recorded:
  member                : 3
  model                 : ResNet50
  python_version        : 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
  platform              : Linux-6.6.122+-x86_64-with-glibc2.39
  tensorflow_version    : 2.21.0
  keras_version         : 3.13.2
  numpy_version         : 2.1.3
  pandas_version        : 2.2.3
  scikit_learn_version  : 1.6.1
  kagglehub_version     : 1.0.2
  matplotlib_version    : 3.10.0
  gpu_name              : No GPU detected
  nvidia_smi            : N/A
  seed                  : 42
  determinism_status    : TensorFlow op determinism enabled


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path

# Folder layout expected inside Dataset:
#   Dataset/ga/correct, Dataset/ga/wrong, Dataset/ra/correct, ... (5 letters x correct/wrong)
PROJECT_DIR = Path("/content/drive/MyDrive/dataset")
DATA_DIR = Path("/content/drive/MyDrive/dataset")
SPLITS_DIR = PROJECT_DIR / "data" / "splits_10class"
RESULTS_DIR = PROJECT_DIR / "results_10class"
PLOTS_DIR = RESULTS_DIR / "plots"
METRICS_DIR = RESULTS_DIR / "metrics"
MODELS_DIR = PROJECT_DIR / "models_10class"
FINAL_MODEL_DIR = MODELS_DIR / "final"
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"

for folder in [
    SPLITS_DIR, PLOTS_DIR, METRICS_DIR, FINAL_MODEL_DIR, CHECKPOINT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

## 1. Scan the dataset (`correct` / `wrong`)

Expected layout (inside `DATA_DIR`):
```
raw/
  correct/  img1.jpg, img2.png, ...
  wrong/    img1.jpg, ...
```
This cell lists every image, checks that it opens, removes exact duplicates (so the same image can't end up in both train and test), and builds a table of `path / label / class`.

In [6]:
import io
from PIL import Image

# Image settings
IMG_HEIGHT = 224          # ResNet50 standard input size
IMG_WIDTH  = 224
BATCH_SIZE = 32

LETTERS = ["ga", "ra", "ta", "u", "la"]          # folder names of the 5 letters
STATES  = ["correct", "wrong"]                   # sub-folder names inside each letter
CLASS_NAMES = [f"{l}_{s}" for l in LETTERS for s in STATES]   # 10 classes -> labels 0..9
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".bmp"}

if not DATA_DIR.exists():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}  -> check the path in the paths cell")

rows, corrupt = [], []
for letter in LETTERS:
    for state in STATES:
        cls = f"{letter}_{state}"
        cls_dir = DATA_DIR / letter / state
        if not cls_dir.is_dir():
            raise FileNotFoundError(
                f"Missing folder: {cls_dir}\nFolders found in DATA_DIR: {[p.name for p in DATA_DIR.iterdir()]}"
            )
        for p in sorted(cls_dir.rglob("*")):
            if p.suffix.lower() not in IMAGE_EXTS:
                continue
            try:
                data = p.read_bytes()
                Image.open(io.BytesIO(data)).verify()          # raises if the file is broken
                rows.append({"path": str(p), "label": CLASS_TO_ID[cls], "class": cls,
                             "letter": letter, "state": state,
                             "md5": hashlib.md5(data).hexdigest()})
            except Exception as e:
                corrupt.append((str(p), str(e)))

df = pd.DataFrame(rows)
print(f"Valid images found : {len(df)}")
print(f"Corrupt/unreadable : {len(corrupt)}")
for path, err in corrupt[:5]:
    print("   ", path, "->", err)

# Same image present in DIFFERENT folders = conflicting label -> remove all copies
n_labels_per_hash = df.groupby("md5")["label"].nunique()
conflict_hashes = n_labels_per_hash[n_labels_per_hash > 1].index
if len(conflict_hashes):
    print(f"WARNING: {len(conflict_hashes)} image(s) appear in more than one folder -> removed")
    df = df[~df["md5"].isin(conflict_hashes)]

# Exact duplicates inside the same class -> keep one copy
n_before = len(df)
df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)
print(f"Exact duplicates removed: {n_before - len(df)}")

print("\nImages per class:")
print(df["class"].value_counts().reindex(CLASS_NAMES).to_string())

Valid images found : 1001
Corrupt/unreadable : 0
Exact duplicates removed: 2

Images per class:
class
ga_correct    113
ga_wrong       88
ra_correct    125
ra_wrong       74
ta_correct     79
ta_wrong      121
u_correct      65
u_wrong       135
la_correct     63
la_wrong      136


## 2. Train / validation / test split (70 / 15 / 15, stratified)

The split is saved as CSV files so every experiment (and every team member) uses exactly the same images.

In [7]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_csv(SPLITS_DIR / f"{name}.csv", index=False)
    counts = part["class"].value_counts().to_dict()
    print(f"{name:<5}: {len(part):>5} images  {counts}")

train:   699 images  {'la_wrong': 95, 'u_wrong': 94, 'ra_correct': 87, 'ta_wrong': 85, 'ga_correct': 79, 'ga_wrong': 62, 'ta_correct': 55, 'ra_wrong': 52, 'u_correct': 46, 'la_correct': 44}
val  :   150 images  {'la_wrong': 21, 'u_wrong': 21, 'ra_correct': 19, 'ta_wrong': 18, 'ga_correct': 17, 'ga_wrong': 13, 'ta_correct': 12, 'ra_wrong': 11, 'u_correct': 9, 'la_correct': 9}
test :   150 images  {'u_wrong': 20, 'la_wrong': 20, 'ra_correct': 19, 'ta_wrong': 18, 'ga_correct': 17, 'ga_wrong': 13, 'ta_correct': 12, 'ra_wrong': 11, 'u_correct': 10, 'la_correct': 10}
